# 03 - Train Cross-Encoder NLI on cleaned full AllNLI with early stopping

Notebook nay chay doc lap tren Kaggle. Cross-Encoder train tren clean train split day du, dung validation macro-F1 de early stopping, roi danh gia test 5k va thoi gian scoring test 5k.

In [1]:
from pathlib import Path

PROJECT_ROOT = Path('/kaggle/working/similarity_search')
GITHUB_REPOSITORY_URL = 'https://github.com/PhDQuang/similarity_search.git'

if not PROJECT_ROOT.exists():
    !git clone {GITHUB_REPOSITORY_URL} {PROJECT_ROOT}

%cd {PROJECT_ROOT}
%pip install -q -r fix/requirements-kaggle.txt
%pip install -q -e fix

Cloning into '/kaggle/working/similarity_search'...
remote: Enumerating objects: 222, done.
remote: Counting objects: 100% (222/222), done.
remote: Compressing objects: 100% (155/155), done.
remote: Total 222 (delta 90), reused 185 (delta 53), pack-reused 0 (from 0)
Receiving objects: 100% (222/222), 481.65 KiB | 9.83 MiB/s, done.
Resolving deltas: 100% (90/90), done.
/kaggle/working/similarity_search
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for similarity-search-fix (pyproject.toml) ... done
Note: you

In [2]:
from pathlib import Path
import shutil

CLEAN_DATA_DIR = Path('fix/data/processed/allnli_70_15_15_clean/pair-class')
KAGGLE_CLEAN_CANDIDATES = [
    Path('/kaggle/input/allnli-70-15-15-clean/pair-class'),
    Path('/kaggle/input/allnli-70-15-15-clean/allnli_70_15_15_clean/pair-class'),
]

def has_clean_data(path: Path) -> bool:
    return all((path / f'{split}.parquet').exists() for split in ('train', 'val', 'test'))

if not has_clean_data(CLEAN_DATA_DIR):
    source = next((path for path in KAGGLE_CLEAN_CANDIDATES if has_clean_data(path)), None)
    if source is not None:
        CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)
        for item in source.iterdir():
            if item.is_file():
                shutil.copy2(item, CLEAN_DATA_DIR / item.name)
    else:
        !python -m similarity_search_fix.data.prepare_allnli_70_15_15_clean --output-dir {CLEAN_DATA_DIR} --seed 42

assert has_clean_data(CLEAN_DATA_DIR), f'Missing clean data: {CLEAN_DATA_DIR}'
print('Using clean data:', CLEAN_DATA_DIR)

README.md: 5.15kB [00:00, 2.32MB/s]
pair-class/train-00000-of-00001.parquet: 100%|█| 69.5M/69.5M [00:03<00:00, 22.8M
pair-class/dev-00000-of-00001.parquet: 100%|█| 1.57M/1.57M [00:00<00:00, 3.80MB/
pair-class/test-00000-of-00001.parquet: 100%|█| 1.61M/1.61M [00:00<00:00, 2.62MB
Generating train split: 100%|█| 942069/942069 [00:00<00:00, 1301758.89 examples/
Generating test split: 100%|██| 19656/19656 [00:00<00:00, 1152182.12 examples/s]
{
  "dataset_name": "sentence-transformers/all-nli",
  "dataset_config": "pair-class",
  "created_at_utc": "2026-07-05T14:55:30.535398+00:00",
  "seed": 42,
  "split_ratios": {
    "train": 0.7,
    "val": 0.15,
    "test": 0.15
  },
  "saved_paths": {
    "train": "fix/data/processed/allnli_70_15_15_clean/pair-class/train.parquet",
    "val": "fix/data/processed/allnli_70_15_15_clean/pair-class/val.parquet",
    "test": "fix/data/processed/allnli_70_15_15_clean/pair-class/test.parquet"
  },
  "row_counts": {
    "train": 686442,
    "val": 147033,
    

In [3]:
from pathlib import Path

OUTPUT_DIR = Path('/kaggle/working/fix_outputs/cross_encoder_clean_full')
MODEL_DIR = Path('/kaggle/working/fix_models/cross_encoder_clean_full')

!python -m similarity_search_fix.models.train_cross_encoder \
  --input-dir {CLEAN_DATA_DIR} \
  --output-dir {OUTPUT_DIR} \
  --model-dir {MODEL_DIR} \
  --num-train-epochs 5 \
  --batch-size 32 \
  --eval-batch-size 64 \
  --learning-rate 2e-5 \
  --eval-steps 1000 \
  --save-steps 1000 \
  --trainer-eval-samples 20000 \
  --early-stopping-patience 2 \
  --early-stopping-threshold 0.0 \
  --max-retrieval-queries 1000 \
  --test-sample-size 5000 \
  --seed 42

tokenizer_config.json: 100%|██████████████████| 48.0/48.0 [00:00<00:00, 354kB/s]
config.json: 100%|█████████████████████████████| 483/483 [00:00<00:00, 4.81MB/s]
vocab.txt: 232kB [00:00, 10.2MB/s]
tokenizer.json: 466kB [00:00, 23.6MB/s]
Map: 100%|████████████████████| 146965/146965 [00:09<00:00, 15199.00 examples/s]
model.safetensors: 100%|██████████████████████| 268M/268M [00:02<00:00, 132MB/s]
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
  0%|                                                 | 0/53630 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will ins

In [4]:
from pathlib import Path
import json
import shutil

ARTIFACT_DIR = Path('/kaggle/working/artifacts_cross_encoder_clean_full')
if ARTIFACT_DIR.exists():
    shutil.rmtree(ARTIFACT_DIR)
ARTIFACT_DIR.mkdir(parents=True)
shutil.copytree(OUTPUT_DIR, ARTIFACT_DIR / 'outputs')
shutil.copytree(MODEL_DIR, ARTIFACT_DIR / 'model')
zip_path = shutil.make_archive(str(ARTIFACT_DIR), 'zip', ARTIFACT_DIR)
print('Download artifact:', zip_path)

display(json.loads((OUTPUT_DIR / 'metrics.json').read_text()))
display(json.loads((OUTPUT_DIR / 'test5k_performance.json').read_text()))

Download artifact: /kaggle/working/artifacts_cross_encoder_clean_full.zip


{'task': 'cross-encoder-nli-reranker-for-semantic-search',
 'fixed_dataset': 'AllNLI pair-class full 70/15/15',
 'positive_label': 'entailment',
 'model': {'name': 'Cross-Encoder NLI',
  'base_model': 'distilbert-base-uncased',
  'trained_in_project': True,
  'loss': 'CrossEntropyLoss',
  'num_labels': 3},
 'nli': {'val': {'val_accuracy': 0.8478096753789965,
   'val_macro_f1': 0.8478997819707651,
   'val_entailment_f1': 0.8621577677011483},
  'test': {'test_accuracy': 0.8472969754703501,
   'test_macro_f1': 0.8473275406185085,
   'test_entailment_f1': 0.8633817909783862}},
 'threshold_selection': {'split': 'val',
  'threshold': 0.43178221583366394,
  'best_f1': 0.8635359781561195},
 'pair_classification': {'val': {'threshold': 0.43178221583366394,
   'accuracy': 0.9075445648255833,
   'precision': 0.8490800694883134,
   'recall': 0.8784926470588236,
   'f1': 0.8635359781561195,
   'average_precision': 0.9265339927854036,
   'mean_positive_score': 0.8240587115287781,
   'mean_negative_s

{'sample_rows': 5000,
 'elapsed_seconds': 12.97889240800032,
 'pairs_per_second': 385.2408851866242,
 'threshold': 0.43178221583366394,
 'pair_classification': {'threshold': 0.43178221583366394,
  'accuracy': 0.9074,
  'precision': 0.8536443148688047,
  'recall': 0.8735083532219571,
  'f1': 0.8634621055735772,
  'average_precision': 0.9313303052085753,
  'mean_positive_score': 0.8196788430213928,
  'mean_negative_score': 0.08217062056064606,
  'roc_auc': 0.965967118432805}}